# Clase 3 — Demo: Pydantic en acción

**IADS 3 | Desarrollo de Sistemas de IA — Unidad 2**

Esta demo **no necesita ninguna API key**. En vez de llamar a un LLM real, simulamos las respuestas que *podría* devolver un modelo — algunas bien formadas, otras rotas a propósito — para ver a Pydantic aceptarlas o rechazarlas en tiempo real.

La idea: separar dos cosas que se mezclan fácil — **¿el LLM respondió?** vs **¿lo que respondió sirve para tocar la base de datos de Ortelana?**

## Paso 1: instalar Pydantic

Colab no lo trae por defecto en todas las versiones — esta celda lo asegura.

In [ ]:
!pip install pydantic --quiet

## Paso 2: el contrato de datos (el mismo `schemas.py` de la clase)

In [ ]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Literal, Optional


IntencionesPermitidas = Literal[
    "ALTA_DISTRIBUIDOR", "CONSULTA_DISTRIBUIDOR", "CONSULTA_STOCK", "RECLAMO_OPERATIVO"
]


class ConsultaClienteSchema(BaseModel):
    """
    Contrato de datos estricto para normalizar consultas
    recibidas por Ortelana Textil.
    """
    intencion_principal: IntencionesPermitidas = Field(
        description="Categoría raíz del mensaje según el árbol de intenciones del negocio."
    )
    identificador_fiscal: Optional[str] = Field(
        default=None,
        description="Identificador fiscal numérico (ej: CUIT de 11 dígitos), sin guiones ni espacios."
    )
    razon_social_declarada: Optional[str] = Field(
        default=None,
        description="Nombre de la empresa o razón social provista por el remitente."
    )
    tipo_tela: Optional[str] = Field(
        default=None,
        description="Nombre comercial de la tela solicitada (ej: gabardina, lona, denim)."
    )
    color: Optional[str] = Field(
        default=None,
        description="Color específico solicitado por el cliente."
    )
    confianza_clasificacion: float = Field(
        description="Nivel de certeza probabilística del modelo en la clasificación, de 0.0 a 1.0."
    )

    @field_validator("identificador_fiscal")
    @classmethod
    def validar_y_limpiar_idfiscal(cls, v):
        if v is None:
            return v
        idfiscal_limpio = "".join(filter(str.isdigit, v))
        if idfiscal_limpio and len(idfiscal_limpio) != 11:
            raise ValueError("El identificador fiscal debe contener exactamente 11 dígitos numéricos.")
        return idfiscal_limpio


print("Esquema definido. Pydantic ya sabe qué forma tiene que tener una consulta válida de Ortelana.")

## Paso 3: simulamos 5 "respuestas del LLM"

En una app real, estos diccionarios vendrían de `completion.choices[0].message.parsed`. Acá los escribimos a mano — algunos perfectos, otros rotos — para ver qué hace Pydantic con cada uno.

In [ ]:
respuestas_simuladas_del_llm = [
    {
        "caso": "1. Todo perfecto",
        "data": {"intencion_principal": "CONSULTA_STOCK", "identificador_fiscal": "20-30456789-9", "razon_social_declarada": "Confecciones del Sur", "tipo_tela": "gabardina", "color": "azul", "confianza_clasificacion": 0.95},
    },
    {
        "caso": "2. Identificador fiscal con menos dígitos de los que debería",
        "data": {"intencion_principal": "ALTA_DISTRIBUIDOR", "identificador_fiscal": "203045678", "razon_social_declarada": "Modas del Sur", "tipo_tela": None, "color": None, "confianza_clasificacion": 0.88},
    },
    {
        "caso": "3. El LLM inventó una intención que no existe en el mapa",
        "data": {"intencion_principal": "CANCELAR_CUENTA", "identificador_fiscal": "20304567899", "razon_social_declarada": None, "tipo_tela": None, "color": None, "confianza_clasificacion": 0.7},
    },
    {
        "caso": "4. Confianza vino como texto en vez de número",
        "data": {"intencion_principal": "CONSULTA_STOCK", "identificador_fiscal": None, "razon_social_declarada": None, "tipo_tela": "denim", "color": None, "confianza_clasificacion": "muy alta"},
    },
    {
        "caso": "5. Sin identificador fiscal (válido: es opcional para una consulta de stock)",
        "data": {"intencion_principal": "CONSULTA_STOCK", "identificador_fiscal": None, "razon_social_declarada": None, "tipo_tela": "jersey", "color": None, "confianza_clasificacion": 0.9},
    },
]

for item in respuestas_simuladas_del_llm:
    print(f"\n--- {item['caso']} ---")
    print(f"Lo que 'devolvió' el LLM: {item['data']}")
    try:
        pedido_validado = ConsultaClienteSchema(**item["data"])
        print(f"✅ ACEPTADO por Pydantic: {pedido_validado}")
    except ValidationError as e:
        print(f"❌ RECHAZADO por Pydantic:")
        for error in e.errors():
            print(f"   - Campo '{error['loc'][0]}': {error['msg']}")

### Para discutir en clase, caso por caso

- **Caso 1 y 5** pasan sin problema — tienen la forma correcta.
- **Caso 2** se rechaza por el validador personalizado que escribimos (`validar_y_limpiar_idfiscal`) — no por Pydantic "de fábrica", sino por una regla que **nosotros** agregamos.
- **Caso 3** se rechaza porque `Literal[...]` (vía `IntencionesPermitidas`) no permite valores fuera de la lista cerrada. Esto es justo la pregunta que quedó abierta en la Clase 2: ¿qué pasa si el LLM detecta una intención que no diseñamos? Acá lo vemos: Pydantic la frena antes de que llegue a ningún lado.
- **Caso 4** se rechaza porque `confianza_clasificacion` espera un `float`, no un texto. Pydantic V2 intenta convertir tipos razonables (`"0.9"` → `0.9`), pero `"muy alta"` no es convertible — y ahí falla.

## Paso 4: ¿qué pasaría sin Pydantic?

Para que quede el contraste bien claro, así se vería el Caso 2 (CUIT corto) si el sistema **no tuviera** esta capa de validación — el dato malo pasaría directo:

In [ ]:
dato_sin_validar = {"intencion_principal": "ALTA_DISTRIBUIDOR", "identificador_fiscal": "203045678", "confianza_clasificacion": 0.88}

print("Sin Pydantic, esto seguiría de largo hacia el ERP real de Ortelana:")
print(dato_sin_validar)
print("\n...con un identificador fiscal de 9 dígitos en vez de 11. El ERP podría aceptarlo mal, o romperse más adelante.")
print("Con Pydantic, como vimos arriba, esto nunca sale del Bloque de validación.")

> **Spoiler de lo que viene:** todo esto validamos con datos que escribimos a mano. En el Bloque 3/4 de hoy conectamos esto mismo a una API real (OpenAI/Anthropic/Gemini), para que el LLM genere estos diccionarios automáticamente a partir de un correo real de un distribuidor — y Pydantic sigue haciendo exactamente este mismo trabajo de guardia.